# 01 | Environment and Framework Overview

This series is written for readers who are new to tensor networks, quantum-inspired machine learning, or `tneq-qc` itself.

By the end of this notebook, you will be able to:

1. explain what problem `tneq-qc` solves;
2. distinguish qubits, cores, bonds, and contractions;
3. create a PyTorch backend and contract a minimal network;
4. follow the rest of the tutorial in a deliberate order.

> The repository currently has no `pyproject.toml` or working dependency manifest. Start Jupyter from the repository root and make sure `numpy`, `torch`, `opt_einsum`, `safetensors`, and `tqdm` are installed. JAX and plotting packages are optional.


## 1. What does this framework do?

A conventional neural network organizes parameters into matrices and layers. A tensor network represents a potentially enormous high-order tensor as a collection of smaller **core tensors** connected by shared indices.

The main `tneq-qc` pipeline is:

```text
ASCII topology
    ↓
QCTN parses connections and infers core shapes
    ↓
TNTensor stores tensor × scale and exposes gradients
    ↓
A contraction strategy builds an execution plan
    ↓
PyTorch or JAX performs numerical work and autodiff
    ↓
A loss and optimizer train the selected cores
```

Four terms appear throughout the project:

- **qubit**: one row of the ASCII graph and one logical wire;
- **core**: a local tensor, shown as a letter in the graph;
- **bond**: a shared index between cores; its number is the index dimension;
- **contraction**: summation over shared indices to combine tensors.


In [1]:
from pathlib import Path
import sys

# This works whether Jupyter starts in the repository root or in notebooks/.
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "tneq_qc").is_dir() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/yuch3n/Documents/Code/Github/tneq-qc


In [2]:
# Check dependencies before importing the framework.
import importlib.util

required = ["numpy", "torch", "opt_einsum", "safetensors", "tqdm"]
optional = ["jax", "matplotlib", "pytest"]

for name in required:
    status = "installed" if importlib.util.find_spec(name) else "MISSING"
    print(f"[required] {name:12s} {status}")
for name in optional:
    status = "installed" if importlib.util.find_spec(name) else "not installed"
    print(f"[optional] {name:12s} {status}")


[required] numpy        installed
[required] torch        installed
[required] opt_einsum   installed
[required] safetensors  installed
[required] tqdm         installed
[optional] jax          installed
[optional] matplotlib   installed
[optional] pytest       installed


## 2. Create a compute backend

A backend owns the actual tensor operations. PyTorch on CPU is the best starting point because it is easy to inspect and is the primary backend used by the repository tests.

Use `float32` for basic experiments. Switch to `complex64` when you need complex amplitudes or conjugate-transpose behavior.


In [3]:
from tneq_qc import BackendFactory

backend = BackendFactory.create_backend(
    "pytorch",
    device="cpu",
    dtype="float32",
)
print(backend.get_backend_info())


BackendInfo(backend_type='pytorch', device='cpu', dtype='float32', config={'enable_auto_scale': False})


## 3. Build and contract the smallest useful network

The same symbol `A` appears on both rows, so it denotes one core spanning two qubits. Each row supplies a left and right edge of dimension 2, giving `A` the shape `(2, 2, 2, 2)`.


In [4]:
from tneq_qc import QCTN, EngineCommon

graph = """\
-2-A-2-
-2-A-2-"""

network = QCTN(graph, backend=backend).auto_init()
print(network)
print("Number of qubits:", network.nqubits)
print("Number of cores:", network.ncores)
print("Shape of A:", network["A"].shape)


QCTN(nqubits=2, cores=[A(2, 2, 2, 2)])
Number of qubits: 2
Number of cores: 1
Shape of A: torch.Size([2, 2, 2, 2])


In [5]:
engine = EngineCommon(backend=backend, strategy="row_priority")
result = engine.contract(network)

print("Result type:", type(result).__name__)
print("Result shape:", result.shape)
print("First effective values:", result.numpy().reshape(-1)[:8])


[Compiler] Strategy candidates: ['row_priority'], Testing 1 strategies...
  [row_priority] Compatibility: True
  [row_priority] Estimated cost: 5.00e+05 FLOPs
[Compiler] Selected strategy: row_priority (cost: 5.00e+05)
Result type: TNTensor
Result shape: torch.Size([2, 2, 2, 2])
First effective values: [ 0.03689738 -0.2751608   0.68426865 -0.5940145   1.0888237   1.4164112
 -0.88774794  0.8695292 ]


## 4. How to interpret the result

This graph leaves input and output edges open, so the result is not necessarily a scalar. A contraction becomes scalar only when states, measurements, or trace operations close every external edge. With batched measurement tensors, the final result is usually one scalar per batch item.

Compiler messages during the first contraction are expected. The engine compiles and caches the contraction function on the QCTN instance; later calls reuse it.

## 5. Recommended path through the series

1. **02** — learn how ASCII graphs determine shapes;
2. **03** — understand scale, batches, references, and gradients;
3. **04** — compose modules and choose contraction strategies;
4. **05** — write a complete TNEQ training loop;
5. **06** — inject ordinary data into a BornMachine;
6. **07** — calculate probabilities and generate samples;
7. **08** — save models and extend the framework.

### Exercises

- Change the backend dtype to `complex64` and inspect the output dtype.
- Add a third row containing `A` and predict its new tensor rank.
- Run `engine.contract(network)` twice and compare the compiler logs.
